In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset

In [3]:
dataset =load_dataset("CShorten/ML-ArXiv-Papers",split='train')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
import pandas as pd

In [7]:
df=pd.DataFrame(dataset)
df

,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...,...,...
117587,4995,NaN,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,4996,NaN,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,4997,NaN,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,4998,NaN,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [9]:
df=df[['title','abstract']]

In [10]:
df

,title,abstract
0,Learning from compressed observations,The problem of statistical learning is to co...
1,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...
...,...,...
117587,Detecting COVID-19 Conspiracy Theories with Tr...,The sharing of fake news and conspiracy theori...
117588,Fair Feature Subset Selection using Multiobjec...,The feature subset selection problem aims at s...
117589,A Simple Duality Proof for Wasserstein Distrib...,We present a short and elementary proof of the...
117590,Combined Learning of Neural Network Weights fo...,"We introduce CoLN, Combined Learning of Neural..."


In [11]:
df.shape

(117592, 2)

In [12]:
df = df.head(15000)
df.shape

(15000, 2)

In [13]:
df.isnull().sum()

,0
title,0
abstract,0


In [14]:
df["paper_text"] = df["title"]+" "+df["abstract"]

/tmp/ipykernel_4528/1043561184.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"] = df["title"]+" "+df["abstract"]


In [15]:
df[["paper_text"]].head()

,paper_text
0,Learning from compressed observations The pr...
1,Sensor Networks with Random Links: Topology De...
2,The on-line shortest path problem under partia...
3,A neural network approach to ordinal regressio...
4,Parametric Learning and Monte Carlo Optimizati...


In [19]:
from sentence_transformers import SentenceTransformer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [20]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [22]:
df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
df["paper_text"]=df["paper_text"].str.strip()

/tmp/ipykernel_4528/2190359946.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.replace("\n"," ",regex=False)
/tmp/ipykernel_4528/2190359946.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["paper_text"]=df["paper_text"].str.strip()


In [28]:
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
similarity = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[0].reshape(1,-1))
print(similarity)

[[1.0000001]]


In [30]:
similarity = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[1].reshape(1,-1))
print(similarity)

[[0.36625272]]


In [31]:
for i in range(1,5):
  sim = cosine_similarity(sample_embedding[0].reshape(1,-1),sample_embedding[i].reshape(1,-1))
  print(sim)

[[0.36625272]]
[[0.33522844]]
[[0.15505108]]
[[0.37421533]]


Generating Full Embedding


In [32]:
import os
import numpy as np
if os.path.exists("paper_embeddings.npy"):
    print("Loading saved embeddings")
    embeddings = np.load("paper_embeddings.npy")
else:
    print("Generating embeddings")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save("paper_embeddings.npy", embeddings)
    print("Embeddings saved successfully!")

Loading saved embeddings


In [35]:
!pip install faiss-cpu


In [36]:
import faiss

In [37]:
if os.path.exists("paper_faiss.index"):
    print("Loading existing FAISS index")
    index = faiss.read_index("paper_faiss.index")
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    faiss.write_index(index, "paper_faiss.index")
    print("FAISS index saved successfully!")

Loading existing FAISS index


In [39]:
query = "deep learning for medical image analysis"
query_embedding = model.encode([query])
query_embedding.shape

(1, 384)

In [40]:
faiss.normalize_L2(query_embedding)

In [41]:
D,I = index.search(query_embedding,5)
print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [42]:
print(df.iloc[10466]["title"])

A Perspective on Deep Imaging


In [43]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [44]:
print(df.iloc[11873]["title"])

Classification of MRI data using Deep Learning and Gaussian
  Process-based Model Selection


In [45]:
def search_paper(query,k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I = index.search(query_embedding,k)
  return D,I

In [46]:
D,I = search_paper("deep learning for medical image analysis")
print(D)
print(I)

[[0.6807244  0.67092204 0.65219975 0.62811744 0.61311525]]
[[10466 13730 11873 12691 11282]]


In [47]:
def search_paper(query,k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D,I = index.search(query_embedding,k)
  for score,idx in zip(D[0],I[0]):
    print("Similarity Score :",score)
    print("Title :",df.iloc[idx]["title"])
    print("Abstract :",df.iloc[idx]["abstract"][:500])
    print()

In [48]:
search_paper("deep learning for medical image analysis")

Similarity Score : 0.6807244
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

Similarity Score : 0.67092204
Title : Convolutional Neural Networks for Medical Image Analysis: Full Training
  or Fine Tuning?
Abstract :   Training a deep convolutional neural network (CNN) from scratch is difficult
because it requires a large amount of labeled training data and a great deal of
expertise to ensure proper convergence. A promising alternative is to fine-tune
a CNN that has been pre-trained using, for

In [49]:
!pip install transformers==4.46.3

In [50]:
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model = "facebook/bart-large-cnn",
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [52]:
summary=summarizer(df.iloc[10466]["abstract"],max_length = 120 , min_length = 40)
print(summary[0]["summary_text"])

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.


In [55]:
summary[0]["summary_text"]

'The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.'

In [56]:
for score,idx in zip(D[0],I[0]):
    print("Similarity Score :",score)
    print("Title :",df.iloc[idx]["title"])
    print("Abstract :",df.iloc[idx]["abstract"][:500])
    summary=summarizer(df.iloc[idx]["abstract"],max_length = 120 , min_length = 40)
    print(summary[0]["summary_text"])
    print()

Similarity Score : 0.6807244
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance
The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Similarity Score : 0.67092204
Title : Convolutional Neural Networks for Medical

In [57]:
pip install keybert==0.8.5

In [58]:
from keybert import KeyBERT

In [59]:
kw_model = KeyBERT(model)


In [60]:
def search_and_summarize(query, k=5):
  query_embedding = model.encode([query])
  faiss.normalize_L2(query_embedding)
  D, I = index.search(query_embedding, k)
  for score, idx in zip(D[0], I[0]):
    print("Similarity score :", score)
    print("Title :", df.iloc[idx]["title"])
    print("Abstract :", df.iloc[idx]["abstract"][:500])
    print()
    summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
    print(summary[0]["summary_text"])
    print()
    abstract = df.iloc[idx]["abstract"]
    keywords = kw_model.extract_keywords(abstract, keyphrase_ngram_range=(1,3), stop_words="english")
    print("Keywords:")
    for k, s in keywords:
      if s > 0.5:
        print(" •", k)
    print()

In [61]:
search_and_summarize("Deep learning in medical imaging",k=5)

Similarity score : 0.73549855
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Keywords:
 • tomographic imaging deep
 • imaging deep learning
 • learning me

In [62]:
type(kw_model)

keybert._model.KeyBERT

In [63]:
print(df.iloc[10466]["abstract"])

  The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance in clinical and
preclinical applications. To realize the full impact of machine learning on
medical imaging, major challenges must be addressed.



In [64]:
text = df.iloc[10466]["abstract"]
keywords = kw_model.extract_keywords(text)

In [65]:
print(keywords)

[('imaging', 0.4528), ('tomographic', 0.4488), ('reconstruction', 0.3623), ('deep', 0.3003), ('learning', 0.2622)]


In [68]:
abstract = df.iloc[idx]["abstract"]
keywords=kw_model.extract_keywords(abstract,keyphrase_ngram_range=(1,3),stop_words = "english")

In [69]:
print(keywords)

[('images classification lung', 0.6442), ('images lung nodules', 0.6278), ('classification lung nodules', 0.6152), ('nodules present cnn', 0.608), ('features lung nodules', 0.6006)]


In [70]:
search_and_summarize("Deep learning in medical imaging", k=5)

Similarity score : 0.73549855
Title : A Perspective on Deep Imaging
Abstract :   The combination of tomographic imaging and deep learning, or machine learning
in general, promises to empower not only image analysis but also image
reconstruction. The latter aspect is considered in this perspective article
with an emphasis on medical imaging to develop a new generation of image
reconstruction theories and techniques. This direction might lead to
intelligent utilization of domain knowledge from big data, innovative
approaches for image reconstruction, and superior performance

The combination of tomographic imaging and deep learning, or machine learning, promises to empower not only image analysis but also image reconstructions. This direction might lead to intelligent utilization of domain knowledge from big data, innovativeapproaches for image reconstruction, and superior performance in clinical applications.

Keywords:
 • tomographic imaging deep
 • imaging deep learning
 • learning me

#Named Entity Recognition

In [71]:
import re

In [72]:
frameworks = [
    # Deep Learning Frameworks
    'tensorflow', 'keras', 'pytorch', 'jax', 'mxnet', 'caffe',
    'theano', 'paddlepaddle', 'fastai', 'chainer',

    # ML Libraries
    'scikit-learn', 'xgboost', 'lightgbm', 'catboost', 'statsmodels',

    # NLP Libraries
    'huggingface', 'spacy', 'nltk', 'gensim', 'transformers',

    # Computer Vision
    'opencv', 'pillow', 'torchvision', 'albumentations',

    # Data & Compute
    'numpy', 'pandas', 'scipy', 'cupy', 'dask', 'spark',

    # Search & Retrieval
    'faiss', 'elasticsearch', 'annoy',

    # Web & API
    'flask', 'fastapi', 'streamlit', 'django',

    # Other
    'tensorboard', 'wandb', 'mlflow', 'ray', 'celery'
]

In [73]:
models = ['bert', 'gpt', 'transformer', 'resnet', 'yolo', 'lstm', 'cnn', 'alexnet', 'roberta', 'llama', 'vgg', 'gan', 'rnn', 'efficientnet', 'clip', 'dalle', 'mistral', 'gemini', 'vit', 'unet', 't5', 'gpt2', 'gpt3', 'gpt4', 'wavenet', 'word2vec', 'glove']

In [74]:
programming_languages = ['python', 'r', 'java', 'c++', 'javascript', 'go', 'julia', 'matlab', 'scala', 'rust', 'cuda', 'sql']

In [75]:
organizations = ['google', 'microsoft', 'ibm', 'facebook', 'amazon', 'apple', 'openai', 'meta', 'nvidia', 'deepmind', 'anthropic', 'hugging face', 'stanford', 'mit', 'berkeley', 'oxford', 'cambridge']

In [76]:
entity_descriptions = {
    # Frameworks
    'tensorflow': 'ML framework by Google',
    'pytorch': 'ML framework by Facebook',
    'keras': 'High level neural network API',
    'scikit-learn': 'ML library for Python',
    'huggingface': 'NLP models and datasets platform',
    'opencv': 'Computer vision library',
    'xgboost': 'Gradient boosting framework',
    'faiss': 'Similarity search library by Facebook',
    'spacy': 'NLP library for Python',
    'nltk': 'Natural language toolkit',

    # Models
    'bert': 'Language model by Google for NLP',
    'gpt': 'Language model by OpenAI',
    'resnet': 'Image classification model',
    'cnn': 'Convolutional Neural Network',
    'lstm': 'Long Short Term Memory network',
    'transformer': 'Attention based neural network',
    'yolo': 'Real time object detection model',
    'gan': 'Generative Adversarial Network',
    'vgg': 'Deep CNN for image recognition',
    'unet': 'Image segmentation model',

    # Languages
    'python': 'Popular programming language for ML',
    'r': 'Statistical programming language',
    'java': 'Object oriented programming language',
    'matlab': 'Numerical computing environment',
    'cuda': 'Parallel computing platform by Nvidia',

    # Organizations
    'google': 'AI research and technology company',
    'microsoft': 'Technology and AI company',
    'facebook': 'Social media and AI research company',
    'openai': 'AI safety and research company',
    'nvidia': 'GPU and AI hardware company',
    'deepmind': 'AI research lab by Google',
    'meta': 'AI research and social media company',
    'stanford': 'Leading AI research university',
    'mit': 'Leading technology university',
}

In [77]:
def classify_query_entities(query):
    query_lower = query.lower()
    found_entities = {}
    for fw in frameworks:
        if re.search(r'\b' + re.escape(fw) + r'\b', query_lower):
            desc = entity_descriptions.get(fw, '')
            found_entities[fw] = ('Framework', desc)
    for m in models:
        if re.search(r'\b' + re.escape(m) + r'\b', query_lower):
            desc = entity_descriptions.get(m, '')
            found_entities[m] = ('Model', desc)
    for lang in programming_languages:
        if re.search(r'\b' + re.escape(lang) + r'\b', query_lower):
            desc = entity_descriptions.get(lang, '')
            found_entities[lang] = ('Programming Language', desc)
    for org in organizations:
        if re.search(r'\b' + re.escape(org) + r'\b', query_lower):
            desc = entity_descriptions.get(org, '')
            found_entities[org] = ('Organization', desc)
    return found_entities

In [78]:
entities = classify_query_entities("research using pytorch and BERT from google")
print(entities)

{'pytorch': ('Framework', 'ML framework by Facebook'), 'bert': ('Model', 'Language model by Google for NLP'), 'google': ('Organization', 'AI research and technology company')}


In [79]:
for entity, entity_type in entities.items():
    print(f"- {entity} : {entity_type}")

- pytorch : ('Framework', 'ML framework by Facebook')
- bert : ('Model', 'Language model by Google for NLP')
- google : ('Organization', 'AI research and technology company')


In [80]:
entities = classify_query_entities("Python on medical dataset with fast api")
for entity, entity_type in entities.items():
    print(f"- {entity} : {entity_type}")


- python : ('Programming Language', 'Popular programming language for ML')


In [81]:
entities = classify_query_entities("Python on medical dataset with fastapi")
for entity, entity_type in entities.items():
    print(f"- {entity} : {entity_type}")

- fastapi : ('Framework', '')
- python : ('Programming Language', 'Popular programming language for ML')


In [82]:
def search_and_summarize_with_ner(query, k=5):

    # NER on query
    print("Query :", query)
    print()
    entities = classify_query_entities(query)
    if entities:
        print("Detected Entities:")
        for entity, (entity_type, desc) in entities.items():
            if desc:
                print(f"  - {entity} : {entity_type} → {desc}")
            else:
                print(f"  - {entity} : {entity_type}")
    else:
        print("No technical entities detected")
    print()

    # Search + Summarize + Keywords
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    for score, idx in zip(D[0], I[0]):
        print("Similarity score :", score)
        print("Title :", df.iloc[idx]["title"])
        print("Abstract :", df.iloc[idx]["abstract"][:500])
        print()
        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
        print(summary[0]["summary_text"])
        print()
        abstract = df.iloc[idx]["abstract"]
        keywords = kw_model.extract_keywords(abstract, keyphrase_ngram_range=(1,3), stop_words="english")
        print("Keywords:")
        for k, s in keywords:
            if s > 0.5:
                entities_in_keyword = classify_query_entities(k)
                if entities_in_keyword:
                    tags = ", ".join([f"{e}: {t}" for e, (t, d) in entities_in_keyword.items()])
                    print(f" • {k}  [{tags}]")
                else:
                    print(f" • {k}")
        print("-" * 60)

In [83]:
search_and_summarize_with_ner("research using pytorch and BERT from google", k=3)

Query : research using pytorch and BERT from google

Detected Entities:
  - pytorch : Framework → ML framework by Facebook
  - bert : Model → Language model by Google for NLP
  - google : Organization → AI research and technology company

Similarity score : 0.4507684
Title : A Tour of TensorFlow
Abstract :   Deep learning is a branch of artificial intelligence employing deep neural
network architectures that has significantly advanced the state-of-the-art in
computer vision, speech recognition, natural language processing and other
domains. In November 2015, Google released $\textit{TensorFlow}$, an open
source deep learning software library for defining, training and deploying
machine learning models. In this paper, we review TensorFlow and put it in
context of modern deep learning concepts and s

Deep learning is a branch of artificial intelligence employing deep neuralnetwork architectures. In November 2015, Google released TensorFlow, an open-source deep learning software library. 